In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm
import random

In [ ]:
class Config:   
    data_dir = 'data'
    validation_fraction = 0.15
    train_batch = 32
    valid_batch = 128
    test_batch = 128
    
    # model setup
    input_dim = 96
    input_ch = 1
    num_keypoints = 30
    dropout_rate = 0.2
    fc_dim = 512
    
    # training
    seed = 55
    learning_rate = 0.001
    weight_decay = 0.0001
    epochs = 30
    device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(Config.seed)

In [ ]:
training_data = pd.read_csv(Config.data_dir + '/training.csv')
test_data = pd.read_csv(Config.data_dir + '/test.csv')

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(training_data, test_size=Config.validation_fraction, random_state=Config.seed)

In [ ]:
def preprocess_images(data):
    '''Convert raw images strings into numpy arrays, reshape them to (N, 1, 96, 96) and normalize.'''
    images = data['Image'].apply(lambda x: np.fromstring(x, sep=' '))
    images = np.stack(images.values).reshape(-1, 1, 96, 96) / 255.0
    return images

In [ ]:
def preprocess(data):
    '''
    Preprocess images, replace NaN values in keypoints with 0 and normalize keypoints to [0, 1] range.
    Create a mask for not NaN keypoints.
    '''
    images = preprocess_images(data)

    keypoints = data.drop('Image', axis=1).values.astype(np.float32)
    mask = ~np.isnan(keypoints)

    keypoints = np.nan_to_num(keypoints, nan=0.0) / 96.0

    return images, keypoints, mask

In [ ]:
train_imgs, train_keypoints, train_mask = preprocess(train_df)
val_imgs, val_keypoints, val_mask = preprocess(val_df)

test_imgs = preprocess_images(test_data)

In [ ]:
from torch.utils.data import Dataset

class FacialKeypointsDataset(Dataset):
    def __init__(self, images, keypoints, mask):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.keypoints = torch.tensor(keypoints, dtype=torch.float32)
        self.mask = torch.tensor(mask, dtype=torch.float32)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.keypoints[idx], self.mask[idx]


class TestImagesDataset(Dataset):
    def __init__(self, images):
        self.images = torch.tensor(images, dtype=torch.float32)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx]


In [ ]:
train_dataset = FacialKeypointsDataset(train_imgs, train_keypoints, train_mask)
val_dataset = FacialKeypointsDataset(val_imgs, val_keypoints, val_mask)
test_dataset = TestImagesDataset(test_imgs)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=Config.train_batch, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=Config.valid_batch, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=Config.test_batch, shuffle=False)

In [ ]:
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )

class KeypointCNN(nn.Module):
    def __init__(self, config):
        super(KeypointCNN, self).__init__()
        self.layers = nn.Sequential(
            conv_block(config.input_ch, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 256),
            nn.Flatten(),
            nn.Linear(256 * (config.input_dim // 16) * (config.input_dim // 16), config.fc_dim),
            nn.ReLU(),
            nn.Dropout(config.dropout_rate),
            nn.Linear(config.fc_dim, config.num_keypoints)
        )
    

    def forward(self, x):
        return self.layers(x)

In [ ]:
def masked_mse_loss(preds, targets, mask):
    '''
    Compute the mean squared error loss, considering only the keypoints that are not NaN (as indicated by the mask).
    '''
    loss = (preds - targets) ** 2
    loss = loss * mask
    return loss.sum() / mask.sum().clamp(min=1)

In [ ]:
def train(model, optimizer, criterion, train_loader, val_loader, config, scheduler=None):
    history = {'train_mse': [], 'val_mse': [], 'train_rmse_px': [], 'val_rmse_px': []}
    best_val_mse = float('inf')

    for epoch in range(config.epochs):
        model.train()

        train_pbar = tqdm(train_loader, 'Iterating over train data')
        val_pbar = tqdm(val_loader, 'Iterating over validation data')

        sq_err_sum, valid_count = 0.0, 0.0
        for images, keypoints, mask in train_pbar:
            images = images.to(config.device)
            keypoints = keypoints.to(config.device)
            mask = mask.to(config.device)

            optimizer.zero_grad()
            preds = model(images)
            loss = criterion(preds, keypoints, mask)
            loss.backward()
            optimizer.step()

            sq_err_sum += (((preds - keypoints) ** 2) * mask).sum().item()
            valid_count += mask.sum().item()

        train_mse = sq_err_sum / valid_count

        model.eval()
        sq_err_sum, valid_count = 0.0, 0.0
        with torch.inference_mode():
            for images, keypoints, mask in val_pbar:
                images = images.to(config.device)
                keypoints = keypoints.to(config.device)
                mask = mask.to(config.device)

                preds = model(images)

                sq_err_sum += (((preds - keypoints) ** 2) * mask).sum().item()
                valid_count += mask.sum().item()

        val_mse = sq_err_sum / valid_count

        if scheduler is not None:
            scheduler.step(val_mse)

        history['train_mse'].append(train_mse)
        history['val_mse'].append(val_mse)
        history['train_rmse_px'].append((train_mse ** 0.5) * 96)
        history['val_rmse_px'].append((val_mse ** 0.5) * 96)

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            torch.save(model.state_dict(), 'best_model.pt')

        print(f'Epoch: {epoch+1} TrainMSE: {train_mse:.4f} ValidMSE: {val_mse:.4f} '
              f'TrainRMSE(px): {history["train_rmse_px"][-1]:.2f} ValidRMSE(px): {history["val_rmse_px"][-1]:.2f}')

    print(f'Best ValidRMSE(px): {((best_val_mse ** 0.5) * 96):.2f}')

    return history

In [ ]:
@torch.inference_mode()
def evaluate(model, test_loader, config):
    model.eval()
    all_preds = []
    for images in test_loader:
        images = images.to(config.device)
        preds = model(images)
        all_preds.append(preds.cpu())
    return torch.cat(all_preds, dim=0)

In [ ]:
model = KeypointCNN(Config).to(Config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.learning_rate, weight_decay=Config.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = masked_mse_loss

In [ ]:
history = train(model, optimizer, criterion, train_loader, val_loader, Config)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history['train_rmse_px'][1:], label='Train')
plt.plot(history['val_rmse_px'][1:], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('RMSE Loss')
plt.legend()
plt.show()

In [ ]:
model.eval()
images, keypoints, mask = next(iter(val_loader))
with torch.inference_mode():
    preds = model(images.to(Config.device)).cpu()

n = 6
fig, axes = plt.subplots(2, n // 2, figsize=(12, 8))
for i, ax in enumerate(axes.flat):
    img = images[i, 0].numpy() * 255.0
    true_kp = keypoints[i].numpy() * 96.0
    pred_kp = preds[i].numpy() * 96.0

    ax.imshow(img, cmap='gray')
    ax.scatter(true_kp[0::2], true_kp[1::2], c='lime', s=15, label='true')
    ax.scatter(pred_kp[0::2], pred_kp[1::2], c='red', s=15, label='pred')
    ax.axis('off')

axes.flat[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
test_preds = evaluate(model, test_loader, Config)
test_preds = test_preds.numpy() * 96.0

lookup = pd.read_csv(Config.data_dir + '/IdLookupTable.csv')

feature_names = training_data.drop('Image', axis=1).columns.tolist()
col_index = {name: i for i, name in enumerate(feature_names)}

def get_value(row):
    image_idx = row['ImageId'] - 1
    feature_idx = col_index[row['FeatureName']]
    return test_preds[image_idx, feature_idx]

lookup['Location'] = lookup.apply(get_value, axis=1)
submission = lookup[['RowId', 'Location']]
submission.to_csv('submission.csv', index=False)